# Tokens vs Labels lengths check

In [1]:
from datasets import load_from_disk, DatasetDict

TARGET_LABELS = {
    "PASSWORD", "USERNAME", "EMAIL", "PHONENUMBER",
    "FIRSTNAME", "LASTNAME", "SSN",
    "CREDITCARDNUMBER", "CREDITCARDCVV",
    "IBAN", "ACCOUNTNUMBER", "STREET", "IPV4"
}

splits = load_from_disk("../data/processed/mbert_splits")

for split_name, data in splits.items():
    mismatches = []

    for idx, record in enumerate(data):
        tokens = record["mbert_text_tokens"]
        labels = record["mbert_bio_labels"]

        if len(tokens) != len(labels):
            mismatches.append(idx)

    print(
        f"{split_name}: {len(mismatches)} mismatches "
        f"out of {len(data)} records"
    )


c:\Users\lamaa\OneDrive\سطح المكتب\confidentiality-detector\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train: 0 mismatches out of 34802 records
validation: 0 mismatches out of 4350 records
test: 0 mismatches out of 4349 records


# TOKENIZER CONSISTENCY CHECK

In [2]:
from transformers import AutoTokenizer

MODEL_NAME = "google-bert/bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

for split_name, data in splits.items():
    mismatches = []

    for idx, record in enumerate(data):
        stored_tokens = record["mbert_text_tokens"]
        generated_tokens = tokenizer.tokenize(record["source_text"])

        if stored_tokens != generated_tokens:
            mismatches.append(idx)

    print(
        f"{split_name}: {len(data) - len(mismatches)}/{len(data)} matches"
    )

    if mismatches:
        print(f"First mismatch index: {mismatches[0]}")


Could not reach the Hub ([Errno 11001] getaddrinfo failed). Using cached commit hash for 'google-bert/bert-base-multilingual-cased'.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (615 > 512). Running this sequence through the model will result in indexing errors


train: 34802/34802 matches
validation: 4350/4350 matches
test: 4349/4349 matches


# SEQUENCE LENGTH VALIDATION

In [3]:
MAX_TEXT_TOKENS = 510 # mbert uses 512 and each sequence has two special character

for split_name, data in splits.items():
    lengths = [
        len(record["mbert_text_tokens"]) # number of tokens in each sample of each split
        for record in data
    ]
# extract max lengths
    long_texts = sum( # sums True values cause True = 1, False = 0
        length > MAX_TEXT_TOKENS
        for length in lengths
    )

    print(f"\n{split_name}")
    print(f"Maximum length: {max(lengths)}")
    print(f"Long texts: {long_texts}")
    print(f"Percentage: {long_texts / len(data) * 100:.2f}%")


train
Maximum length: 783
Long texts: 3
Percentage: 0.01%

validation
Maximum length: 683
Long texts: 2
Percentage: 0.05%

test
Maximum length: 170
Long texts: 0
Percentage: 0.00%


# BIO SEQUENCE VALIDATION

In [4]:
bio_issues = []
# every issue is stored as a tuple (split name, sample index, issue type)
for split_name, data in splits.items():
    for idx, record in enumerate(data):
        previous_label = "O"

        for label in record["mbert_bio_labels"]:

            if label == "O":
                previous_label = label
                continue

            if not label.startswith(("B-", "I-")):
                bio_issues.append((split_name, idx, "Invalid format"))

            elif label.startswith("I-"):
                entity = label[2:]

                if previous_label not in (f"B-{entity}", f"I-{entity}"):
                    bio_issues.append((split_name, idx, "Invalid I transition"))

            previous_label = label

print("Total BIO issues:", len(bio_issues))
print("First issues:", bio_issues[:10])


Total BIO issues: 841
First issues: [('train', 104, 'Invalid I transition'), ('train', 152, 'Invalid I transition'), ('train', 162, 'Invalid I transition'), ('train', 361, 'Invalid I transition'), ('train', 416, 'Invalid I transition'), ('train', 494, 'Invalid I transition'), ('train', 512, 'Invalid I transition'), ('train', 599, 'Invalid I transition'), ('train', 726, 'Invalid I transition'), ('train', 802, 'Invalid I transition')]


# BIO ERROR ANALYSIS
The following cell present first sample in issues list

In [5]:
from collections import Counter

print("Total issues:", len(bio_issues))
print("Issue types:", Counter(issue[2] for issue in bio_issues)) # every issue is stored as ("train", 104, "Invalid I transition"), so counter here counts how
# many time each issue occures in the data splits

split_name, idx, _ = bio_issues[0]
record = splits[split_name][idx]
labels = record["mbert_bio_labels"]

print("\nRecord:", split_name, idx)
print("Annotation labels:", [ann["label"] for ann in record["privacy_mask"]])
# finding first I unvalid transition 
for i, label in enumerate(labels):
    if label.startswith("I-"):
        previous = labels[i - 1] if i > 0 else "START"

        if previous not in (f"B-{label[2:]}", f"I-{label[2:]}"):
            print("Position:", i)
            print("Previous:", previous)
            print("Current:", label)
            print("Nearby BIO:", labels[max(0, i-3):i+4])
            break


Total issues: 841
Issue types: Counter({'Invalid I transition': 841})

Record: train 104
Annotation labels: ['PREFIX', 'FIRSTNAME', 'LASTNAME', 'CURRENCYSYMBOL', 'AMOUNT', 'EMAIL', 'PHONENUMBER']
Position: 21
Previous: B-CURRENCYSYMBOL
Current: I-AMOUNT
Nearby BIO: ['O', 'O', 'B-CURRENCYSYMBOL', 'I-AMOUNT', 'O', 'O', 'O']


# TARGET VS EXCLUDED LABELS
We will count issues based on label types

In [6]:
issue_counts = Counter()
target_examples = []

for split_name, data in splits.items():
    for idx, record in enumerate(data):
        labels = record["mbert_bio_labels"]
# focusing on labels that starts with I- only
        for pos, label in enumerate(labels):
            if not label.startswith("I-"):
                continue

            entity = label[2:]
            previous = labels[pos - 1] if pos > 0 else "START"
# counting issues based on current labels 
            if previous not in (f"B-{entity}", f"I-{entity}"):
                issue_counts[entity] += 1

                if entity in TARGET_LABELS and len(target_examples) < 5:
                    target_examples.append(
                        (split_name, idx, pos, previous, label)
                    )

target_issues = sum(
    issue_counts[label] for label in TARGET_LABELS
)

print("Total issues:", sum(issue_counts.values()))
print("Target label issues:", target_issues)
print("Excluded label issues:", sum(issue_counts.values()) - target_issues)

print("\nIssues by label:")
print(issue_counts.most_common())

print("\nTarget examples:")
print(target_examples)


Total issues: 841
Target label issues: 137
Excluded label issues: 704

Issues by label:
[('AMOUNT', 353), ('MASKEDNUMBER', 46), ('ACCOUNTNAME', 28), ('FIRSTNAME', 23), ('CURRENCY', 21), ('JOBAREA', 21), ('JOBTITLE', 20), ('EMAIL', 16), ('USERNAME', 15), ('CREDITCARDNUMBER', 13), ('CITY', 13), ('ACCOUNTNUMBER', 13), ('STATE', 13), ('DATE', 12), ('AGE', 12), ('PHONENUMBER', 12), ('ZIPCODE', 12), ('COUNTY', 10), ('STREET', 9), ('SECONDARYADDRESS', 9), ('CREDITCARDISSUER', 9), ('PHONEIMEI', 9), ('TIME', 8), ('CURRENCYSYMBOL', 8), ('SSN', 8), ('IPV4', 8), ('URL', 7), ('EYECOLOR', 7), ('DOB', 7), ('PASSWORD', 7), ('CREDITCARDCVV', 7), ('GENDER', 6), ('NEARBYGPSCOORDINATE', 6), ('COMPANYNAME', 6), ('BUILDINGNUMBER', 6), ('JOBTYPE', 5), ('IPV6', 5), ('LASTNAME', 5), ('CURRENCYNAME', 5), ('CURRENCYCODE', 5), ('USERAGENT', 5), ('MIDDLENAME', 4), ('ORDINALDIRECTION', 4), ('BITCOINADDRESS', 3), ('PREFIX', 3), ('MAC', 3), ('HEIGHT', 2), ('VEHICLEVIN', 2), ('IP', 2), ('BIC', 2), ('VEHICLEVRM', 1), (

# OFFSET MAPPING VALIDATION
This shows the need for a BIO labels REBUILD

In [7]:
# each item in target examples is stored as ("train", 512, 18, "O", "I-STREET")
for split_name, idx, pos, previous, label in target_examples:
    record = splits[split_name][idx]

    offsets = tokenizer(
        record["source_text"],
        add_special_tokens=False,
        return_offsets_mapping=True
    )["offset_mapping"]

    start, end = offsets[pos]
    previous_end = offsets[pos - 1][1] if pos > 0 else -1 

    matching_spans = [
        ann for ann in record["privacy_mask"]
        if ann["label"] == label[2:]
        and start < ann["end"]
        and end > ann["start"]
    ]

    is_entity_start = any(
        previous_end <= ann["start"]
        for ann in matching_spans
    )

    print(
        f"{split_name} | record {idx} | {label} | "
        f"annotation_match={bool(matching_spans)} | "
        f"entity_start={is_entity_start}"
    )


train | record 512 | I-STREET | annotation_match=False | entity_start=False
train | record 843 | I-FIRSTNAME | annotation_match=False | entity_start=False
train | record 1089 | I-CREDITCARDNUMBER | annotation_match=True | entity_start=True
train | record 1117 | I-USERNAME | annotation_match=True | entity_start=False
train | record 1417 | I-ACCOUNTNUMBER | annotation_match=False | entity_start=False


# TOKEN–ANNOTATION ALIGNMENT DIAGNOSTICS


In [8]:
for split_name, idx, pos, previous, label in target_examples:
    record = splits[split_name][idx]

    offsets = tokenizer(
        record["source_text"],
        add_special_tokens=False,
        return_offsets_mapping=True
    )["offset_mapping"]

    start, end = offsets[pos]
# checks overlap between labels annotations and token offset
    overlapping = [
        (ann["label"], ann["start"], ann["end"])
        for ann in record["privacy_mask"]
        if start < ann["end"] and end > ann["start"]
    ]

    same_label = [
        ann for ann in record["privacy_mask"]
        if ann["label"] == label[2:]
    ]

    print(f"\nRecord: {split_name} {idx}")
    print("Token label:", label)
    print("Token offset:", (start, end))
    print("Overlapping annotations:", overlapping)
    print("Same-label annotation spans:",
          [(a["start"], a["end"]) for a in same_label])
    print("Nearby BIO:",
          record["mbert_bio_labels"][max(0, pos-2):pos+3])



Record: train 512
Token label: I-STREET
Token offset: (74, 76)
Overlapping annotations: []
Same-label annotation spans: [(58, 73)]
Nearby BIO: ['O', 'O', 'I-STREET', 'I-STREET', 'I-STREET']

Record: train 843
Token label: I-FIRSTNAME
Token offset: (6, 7)
Overlapping annotations: []
Same-label annotation spans: [(8, 15)]
Nearby BIO: ['O', 'O', 'I-FIRSTNAME', 'I-FIRSTNAME', 'O']

Record: train 1089
Token label: I-CREDITCARDNUMBER
Token offset: (107, 110)
Overlapping annotations: [('CREDITCARDNUMBER', 107, 123)]
Same-label annotation spans: [(107, 123)]
Nearby BIO: ['O', 'O', 'I-CREDITCARDNUMBER', 'I-CREDITCARDNUMBER', 'I-CREDITCARDNUMBER']

Record: train 1117
Token label: I-USERNAME
Token offset: (97, 98)
Overlapping annotations: [('USERNAME', 91, 108)]
Same-label annotation spans: [(91, 108)]
Nearby BIO: ['O', 'O', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME']

Record: train 1417
Token label: I-ACCOUNTNUMBER
Token offset: (202, 203)
Overlapping annotations: []
Same-label annotation spans: 

# Span-value validation

In [9]:
span_mismatches = 0

for data in splits.values():
    for record in data:
        for ann in record["privacy_mask"]:
            extracted = record["source_text"][ann["start"]:ann["end"]]

            if extracted != ann["value"]:
                span_mismatches += 1

print("Span-value mismatches:", span_mismatches)


Span-value mismatches: 0


# BIO label alignment REBUILD
Based on original annotations instaed of useing mbert_bio_labels

In [10]:
def match_tokens_to_annotations(record):
    text = record["source_text"]
    annotations = record["privacy_mask"]

    encoding = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=False
    )

    offsets = encoding["offset_mapping"]
    token_matches = [] # A list to store annotations overlapes with token offsets

    for start, end in offsets:
        matches = [
            i for i, ann in enumerate(annotations)
            if start < ann["end"] and end > ann["start"]
        ]

        token_matches.append(matches)

    return offsets, token_matches


In [ ]:
def build_bio2_labels(record):
    offsets, token_matches = match_tokens_to_annotations(record)
    annotations = record["privacy_mask"]

    labels = [] # a list for the new labels
    previous_annotation = None

    for pos, ((start, end), matches) in enumerate( # has dict for each pair
        zip(offsets, token_matches) # concatenates values from the two lists, each pair has token offset and its intersection with annotations list
    ):
        # if token doesn't match with any entity
        if not matches:
            labels.append("O")
            previous_annotation = None
            continue

        # if token intersect with more than one entity
        if len(matches) != 1:
            labels.append(-100)
            previous_annotation = None
            continue


        ann_id = matches[0]
        ann = annotations[ann_id]

        
        if start < ann["start"] or end > ann["end"]:
            labels.append(-100)
            previous_annotation = None
            continue


        if ann["label"] not in TARGET_LABELS:
            labels.append(-100)
            previous_annotation = None
            continue

        prefix = "I" if previous_annotation == ann_id else "B"

        labels.append(f"{prefix}-{ann['label']}")
        previous_annotation = ann_id
    # resolves annotations alignment 
    for ann_id, ann in enumerate(annotations):
        if ann["label"] not in TARGET_LABELS:
            continue

        positions = [
            pos for pos, matches in enumerate(token_matches)
            if ann_id in matches
        ]

        valid_labels = {f"B-{ann['label']}", f"I-{ann['label']}"}

        if not positions or any(labels[pos] not in valid_labels for pos in positions):
            for pos in positions:
                labels[pos] = -100


    return labels


# VALIDATING REBUILT LABELS

In [12]:
record = splits["train"][0]

new_labels = build_bio2_labels(record)

print("Tokens:", len(record["mbert_text_tokens"]))
print("New labels:", len(new_labels))
print("Ignored tokens:", new_labels.count(-100))
print("Target tokens:", sum(
    isinstance(label, str) and label != "O"
    for label in new_labels
))


Tokens: 58
New labels: 58
Ignored tokens: 20
Target tokens: 0


# Length Validation

In [13]:
print("First record target annotations:", [
    ann["label"]
    for ann in splits["train"][0]["privacy_mask"]
    if ann["label"] in TARGET_LABELS
])

for split_name, data in splits.items():
    passed = 0
    errors = Counter()

    for record in data:
        try:
            labels = build_bio2_labels(record)

            if len(labels) != len(record["mbert_text_tokens"]):
                errors["Length mismatch"] += 1
            else:
                passed += 1

        except ValueError as e:
            error_type = str(e).split(" at token")[0]
            errors[error_type] += 1

    print(f"\n{split_name}:")
    print("Passed:", passed)
    print("Errors:", dict(errors))


First record target annotations: []

train:
Passed: 34802
Errors: {}

validation:
Passed: 4350
Errors: {}

test:
Passed: 4349
Errors: {}


# Entity Coverage Validation

In [14]:
for split_name, data in splits.items():
    coverage = Counter()

    for record in data:
        labels = build_bio2_labels(record)
        _, token_matches = match_tokens_to_annotations(record)

        for ann_id, ann in enumerate(record["privacy_mask"]):
            if ann["label"] not in TARGET_LABELS:
                continue

            positions = [
                pos for pos, matches in enumerate(token_matches)
                if ann_id in matches
            ]

            expected = {f"B-{ann['label']}", f"I-{ann['label']}"}

            matched = sum(
                labels[pos] in expected
                for pos in positions
            )

            if matched == 0:
                coverage["missing"] += 1
            elif matched < len(positions):
                coverage["partial"] += 1
            else:
                coverage["full"] += 1

    print(f"{split_name}: {dict(coverage)}")


train: {'full': 36853, 'missing': 9}
validation: {'full': 4665, 'missing': 2}
test: {'full': 4615, 'missing': 2}


# BIO Validation

In [23]:
bio_issues = []

for split_name, data in splits.items():
    for idx, record in enumerate(data):
        labels = build_bio2_labels(record)
        previous_label = "O"

        for pos, label in enumerate(labels):

            if label == -100 or label == "O":
                previous_label = "O"
                continue

            if not isinstance(label, str) or not label.startswith(("B-", "I-")):
                bio_issues.append((split_name, idx, pos, label, "Invalid format"))
                previous_label = "O"
                continue

            entity = label[2:]

            if entity not in TARGET_LABELS:
                bio_issues.append((split_name, idx, pos, label, "Unknown entity"))
                previous_label = "O"
                continue

            if label.startswith("I-"):
                if previous_label not in (f"B-{entity}", f"I-{entity}"):
                    bio_issues.append(
                        (split_name, idx, pos, label, "Invalid I transition")
                    )

            previous_label = label

print("Total BIO issues:", len(bio_issues))
print("First issues:", bio_issues[:10])

Total BIO issues: 0
First issues: []


# Label Distrbution

In [15]:

label_names = ["O"] + [
    f"{prefix}-{entity}"
    for entity in sorted(TARGET_LABELS)
    for prefix in ("B", "I")
]

label2id = {
    label: idx for idx, label in enumerate(label_names)
}

id2label = {
    idx: label for label, idx in label2id.items()
}

assert len(label_names) == 27

print("Number of labels:", len(label_names))
print("Label mapping:", label2id)


Number of labels: 27
Label mapping: {'O': 0, 'B-ACCOUNTNUMBER': 1, 'I-ACCOUNTNUMBER': 2, 'B-CREDITCARDCVV': 3, 'I-CREDITCARDCVV': 4, 'B-CREDITCARDNUMBER': 5, 'I-CREDITCARDNUMBER': 6, 'B-EMAIL': 7, 'I-EMAIL': 8, 'B-FIRSTNAME': 9, 'I-FIRSTNAME': 10, 'B-IBAN': 11, 'I-IBAN': 12, 'B-IPV4': 13, 'I-IPV4': 14, 'B-LASTNAME': 15, 'I-LASTNAME': 16, 'B-PASSWORD': 17, 'I-PASSWORD': 18, 'B-PHONENUMBER': 19, 'I-PHONENUMBER': 20, 'B-SSN': 21, 'I-SSN': 22, 'B-STREET': 23, 'I-STREET': 24, 'B-USERNAME': 25, 'I-USERNAME': 26}


# MODEL INPUT PREPARATION

In [ ]:
record = next(
    r for r in splits["train"]
    if any(ann["label"] in TARGET_LABELS for ann in r["privacy_mask"])
)

bio_labels = build_bio2_labels(record)

encoding = tokenizer(
    record["source_text"],
    add_special_tokens=True,
    return_special_tokens_mask=True,
    truncation=False
)

assert encoding["special_tokens_mask"].count(0) == len(bio_labels)

label_iterator = iter(bio_labels)
label_ids = []

for is_special in encoding["special_tokens_mask"]:
    if is_special:
        label_ids.append(-100)
    else:
        label = next(label_iterator)
        label_ids.append(-100 if label == -100 else label2id[label])

assert len(encoding["input_ids"]) == len(label_ids)

print("Input IDs:", len(encoding["input_ids"])) # numbers that represents tokens
print("Attention Mask:", len(encoding["attention_mask"])) # presents positions where the model needs attention to
print("Labels:", len(label_ids)) # numbers presents bio labels
print("Target Tokens:", sum(label > 0 for label in label_ids))


Input IDs: 36
Attention Mask: 36
Labels: 36
Target Tokens: 5


In [ ]:
def prepare_example(record):
    bio_labels = build_bio2_labels(record)

    encoding = tokenizer(
        record["source_text"],
        add_special_tokens=True,
        return_special_tokens_mask=True,
        truncation=False
    )

    special_mask = encoding.pop("special_tokens_mask") # cus we do not need special tokens masks later in training

    assert special_mask.count(0) == len(bio_labels)

    label_ids = []
    label_iterator = iter(bio_labels)

    for is_special in special_mask:
        if is_special:
            label_ids.append(-100)
        else:
            label = next(label_iterator)
            label_ids.append(
                -100 if label == -100 else label2id[label]
            )

    return {
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": label_ids
    }


In [ ]:
I_LABEL_IDS = {
    idx for label, idx in label2id.items()
    if label.startswith("I-")
}

def make_windows(record, max_length=512, overlap=32):
    example = prepare_example(record)

    token_ids = example["input_ids"][1:-1]
    labels = example["labels"][1:-1]

    windows = []
    cursor = 0
    total = len(token_ids)

    while cursor < total:
        start = max(0, cursor - overlap)
        end = min(start + max_length - 2, total)
# Preventing an entity from being split between two windows
        while end < total and end > cursor and labels[end] in I_LABEL_IDS:
            end -= 1

        if end <= cursor:
            raise ValueError("Cannot split without cutting an entity")

        window_labels = labels[start:end].copy()
        window_labels[:cursor-start] = [-100] * (cursor-start)

        windows.append({
            "input_ids": [tokenizer.cls_token_id] + token_ids[start:end] + [tokenizer.sep_token_id],
            "attention_mask": [1] * (end - start + 2),
            "labels": [-100] + window_labels + [-100]
        })

        cursor = end

    return windows


# WINDOW VALIDATION

In [ ]:
long_record = next(
    record for record in splits["train"]
    if len(record["mbert_text_tokens"]) > 510
)

windows = make_windows(long_record)
original = prepare_example(long_record)

print("Number of windows:", len(windows))
print("Window lengths:", [len(w["input_ids"]) for w in windows])

print("Valid lengths:", all(
    len(w["input_ids"]) == len(w["attention_mask"]) == len(w["labels"]) <= 512
    for w in windows
))

original_supervised = sum(label != -100 for label in original["labels"])
window_supervised = sum(
    label != -100
    for window in windows
    for label in window["labels"]
)

print("Supervised tokens preserved:", original_supervised == window_supervised)


Number of windows: 2
Window lengths: [512, 139]
Valid lengths: True
Supervised tokens preserved: True


In [ ]:
def prepare_batch(batch):
    prepared = {
        "input_ids": [],
        "attention_mask": [],
        "labels": []
    }

    for i in range(len(batch["source_text"])):
        record = {
            "source_text": batch["source_text"][i],
            "privacy_mask": batch["privacy_mask"][i]
        }

        windows = make_windows(record)

        for window in windows:
            for key in prepared:
                prepared[key].append(window[key])

    return prepared


In [ ]:
tokenized_splits = DatasetDict({
    name: data.map(
        prepare_batch,
        batched=True,
        batch_size=128,
        remove_columns=data.column_names,
        desc=f"Preparing {name}"
    )
    for name, data in splits.items()
})

for name, data in tokenized_splits.items():
    print(f"{name}: {len(data)} windows")

    assert all(
        len(row["input_ids"]) == len(row["attention_mask"]) == len(row["labels"]) <= 512
        for row in data
    )

print("All windows validated!")


Preparing test: 100%|██████████| 4349/4349 [00:03<00:00, 1145.81 examples/s]


train: 34805 windows
validation: 4352 windows
test: 4349 windows
All windows validated!


In [ ]:
from pathlib import Path

output_path = Path("../data/processed/mbert_tokenized")

output_path.parent.mkdir(parents=True, exist_ok=True)

tokenized_splits.save_to_disk(str(output_path))

print("Saved to:", output_path.resolve())


Saving the dataset (1/1 shards): 100%|██████████| 4349/4349 [00:00<00:00, 182366.51 examples/s]

Saved to: C:\Users\lamaa\OneDrive\سطح المكتب\confidentiality-detector\data\processed\mbert_tokenized
